# Beginner 05 — Least-Privilege Tool Access for Agents

## Enterprise scenario

A travel assistant can search policy, inspect trips, book flights, send itinerary emails, and cancel bookings.

We will start with an unsafe "all tools" architecture and progressively introduce:

- tool catalogs;
- dynamic exposure;
- resource and argument authorization;
- approval gates;
- credential isolation;
- budgets;
- chain controls;
- audit evidence.

> The LLM proposes actions. Trusted application code authorizes and executes them.


In [ ]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from typing import Any, Callable, Optional
import hashlib, json, time, uuid

def now():
    return datetime.now(timezone.utc)


## 1 — The unsafe baseline

In [ ]:
def search_flights(origin, destination):
    return {"flights": ["AC101", "LH493"]}

def book_flight(flight, price):
    return {"booking_id": "B-123", "flight": flight, "price": price}

def cancel_booking(booking_id):
    return {"cancelled": booking_id}

def execute_shell(command):
    return {"executed": command}  # NEVER expose this casually in production

ALL_TOOLS = {
    "search_flights": search_flights,
    "book_flight": book_flight,
    "cancel_booking": cancel_booking,
    "execute_shell": execute_shell,
}

print("Unsafe tool set:", list(ALL_TOOLS))


A travel-search task does not need cancellation or shell execution. Exposing them increases the action surface before authorization even begins.

## 2 — Governed tool catalog

In [ ]:
@dataclass(frozen=True)
class ToolSpec:
    id: str
    effect: str
    risk: str
    handler: Callable
    credential_profile: Optional[str] = None
    approval_required: bool = False
    allowed_agents: tuple[str, ...] = ()
    max_calls_per_task: int = 10

CATALOG = {
    "search_flights": ToolSpec(
        "tool:search_flights", "read", "low", search_flights,
        credential_profile="travel-search-read",
        allowed_agents=("agent:travel-planner",),
        max_calls_per_task=10,
    ),
    "book_flight": ToolSpec(
        "tool:book_flight", "financial-write", "high", book_flight,
        credential_profile="travel-book-limited",
        approval_required=True,
        allowed_agents=("agent:travel-planner",),
        max_calls_per_task=2,
    ),
    "cancel_booking": ToolSpec(
        "tool:cancel_booking", "destructive", "high", cancel_booking,
        credential_profile="travel-cancel-limited",
        approval_required=True,
        allowed_agents=("agent:travel-planner",),
        max_calls_per_task=1,
    ),
    "execute_shell": ToolSpec(
        "tool:execute_shell", "code-execution", "critical", execute_shell,
        allowed_agents=(),
        max_calls_per_task=0,
    ),
}

for name, spec in CATALOG.items():
    print(f"{name:18} effect={spec.effect:16} risk={spec.risk}")


## 3 — Task-scoped grants

In [ ]:
@dataclass(frozen=True)
class TaskGrant:
    task_id: str
    agent: str
    requester: str
    allowed_tools: frozenset[str]
    resources: dict[str, frozenset[str]]
    expires_at: datetime

grant = TaskGrant(
    task_id="task:travel-483",
    agent="agent:travel-planner",
    requester="user:alice",
    allowed_tools=frozenset({"search_flights", "book_flight"}),
    resources={
        "trip": frozenset({"trip:483"}),
    },
    expires_at=now() + timedelta(minutes=30),
)


## 4 — Dynamic tool exposure

In [ ]:
def visible_tools(agent: str, grant: TaskGrant):
    if now() >= grant.expires_at or agent != grant.agent:
        return []
    result = []
    for name in grant.allowed_tools:
        spec = CATALOG.get(name)
        if spec and agent in spec.allowed_agents:
            result.append(name)
    return sorted(result)

print("Tools visible to model:", visible_tools("agent:travel-planner", grant))


`cancel_booking` and `execute_shell` are absent. The model does not need to reason about capabilities it cannot use.

## 5 — Execution-time authorization

In [ ]:
def authorize_tool(agent: str, tool_name: str, grant: TaskGrant):
    if now() >= grant.expires_at:
        return False, "task grant expired"
    if agent != grant.agent:
        return False, "grant bound to another agent"
    if tool_name not in grant.allowed_tools:
        return False, "tool not granted for task"
    spec = CATALOG.get(tool_name)
    if not spec:
        return False, "unknown tool"
    if agent not in spec.allowed_agents:
        return False, "agent not allowed by tool policy"
    return True, "tool invocation allowed"

for name in CATALOG:
    print(name, "->", authorize_tool("agent:travel-planner", name, grant))


Even though discovery was filtered, execution re-checks current authorization.

## 6 — Argument-level authorization

In [ ]:
ALLOWED_AIRLINES = {"AC", "LH", "AA"}
MAX_AUTO_SEARCH_PRICE = 2000
MAX_BOOK_PRICE = 1000

def authorize_arguments(tool_name: str, args: dict[str, Any]):
    if tool_name == "search_flights":
        origin = args.get("origin")
        destination = args.get("destination")
        if not origin or not destination:
            return False, "origin and destination required"
        if origin == destination:
            return False, "origin and destination must differ"
        return True, "arguments allowed"

    if tool_name == "book_flight":
        flight = args.get("flight", "")
        price = args.get("price")
        if not isinstance(price, (int, float)) or price <= 0:
            return False, "invalid price"
        airline = flight[:2]
        if airline not in ALLOWED_AIRLINES:
            return False, "airline not approved"
        if price > MAX_BOOK_PRICE:
            return False, "price exceeds delegated maximum"
        return True, "arguments allowed"

    if tool_name == "cancel_booking":
        if not args.get("booking_id"):
            return False, "booking_id required"
        return True, "arguments allowed"

    return False, "no argument policy for tool"

print(authorize_arguments("book_flight", {"flight": "AC101", "price": 700}))
print(authorize_arguments("book_flight", {"flight": "AC101", "price": 7000}))


## 7 — Syntax-valid does not mean authorized

In [ ]:
model_output = {"flight": "AC101", "price": 9000}

schema_valid = (
    isinstance(model_output.get("flight"), str)
    and isinstance(model_output.get("price"), (int, float))
)
print("Schema valid:", schema_valid)
print("Authorized :", authorize_arguments("book_flight", model_output))


## 8 — Approval bound to exact operation

In [ ]:
@dataclass(frozen=True)
class Approval:
    id: str
    approver: str
    task_id: str
    tool: str
    args_hash: str
    expires_at: datetime

def canonical_hash(args):
    raw = json.dumps(args, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()

booking_args = {"flight": "AC101", "price": 700}
approval = Approval(
    id="approval:837",
    approver="user:manager-bob",
    task_id=grant.task_id,
    tool="book_flight",
    args_hash=canonical_hash(booking_args),
    expires_at=now() + timedelta(minutes=10),
)

def validate_approval(approval, task_id, tool, args):
    if now() >= approval.expires_at:
        return False
    return (
        approval.task_id == task_id
        and approval.tool == tool
        and approval.args_hash == canonical_hash(args)
    )

print("Original:", validate_approval(approval, grant.task_id, "book_flight", booking_args))
print("Mutated :", validate_approval(
    approval, grant.task_id, "book_flight",
    {"flight": "AC101", "price": 900}
))


Changing the amount invalidates the approval. Approval is evidence for a specific operation, not a vague conversation state.

## 9 — Credential profiles stay outside the model

In [ ]:
CREDENTIAL_STORE = {
    "travel-search-read": "SECRET_SEARCH_TOKEN",
    "travel-book-limited": "SECRET_BOOKING_TOKEN",
    "travel-cancel-limited": "SECRET_CANCEL_TOKEN",
}

class CredentialBroker:
    def get_for_tool(self, tool_spec: ToolSpec):
        profile = tool_spec.credential_profile
        if not profile:
            return None
        # Production: mint/federate short-lived credentials instead.
        return CREDENTIAL_STORE[profile]

broker = CredentialBroker()

for name in visible_tools("agent:travel-planner", grant):
    print(name, "uses credential profile:", CATALOG[name].credential_profile)

print("Raw credentials are intentionally not passed to model-facing structures.")


## 10 — Call budgets

In [ ]:
class Budget:
    def __init__(self):
        self.calls = {}

    def consume(self, task_id, tool_name, maximum):
        key = (task_id, tool_name)
        used = self.calls.get(key, 0)
        if used >= maximum:
            return False
        self.calls[key] = used + 1
        return True

budget = Budget()

for i in range(4):
    ok = budget.consume(
        grant.task_id,
        "book_flight",
        CATALOG["book_flight"].max_calls_per_task,
    )
    print("attempt", i + 1, "->", "ALLOW" if ok else "DENY")


## 11 — Secure tool gateway

In [ ]:
AUDIT = []

class ToolGateway:
    def __init__(self, catalog, broker, budget):
        self.catalog = catalog
        self.broker = broker
        self.budget = budget

    def invoke(self, *, agent, grant, tool_name, args, approval=None):
        allowed, reason = authorize_tool(agent, tool_name, grant)
        if not allowed:
            return self._deny(grant, agent, tool_name, args, reason)

        spec = self.catalog[tool_name]

        allowed, reason = authorize_arguments(tool_name, args)
        if not allowed:
            return self._deny(grant, agent, tool_name, args, reason)

        if spec.approval_required:
            if not approval or not validate_approval(
                approval, grant.task_id, tool_name, args
            ):
                return self._deny(
                    grant, agent, tool_name, args,
                    "valid bound approval required"
                )

        if not self.budget.consume(
            grant.task_id, tool_name, spec.max_calls_per_task
        ):
            return self._deny(grant, agent, tool_name, args, "call budget exceeded")

        credential = self.broker.get_for_tool(spec)
        # credential would be injected into the downstream client here,
        # never returned to the model.

        result = spec.handler(**args)
        self._audit(grant, agent, tool_name, args, "allow", "all controls passed")
        return {"status": "ok", "result": result}

    def _deny(self, grant, agent, tool, args, reason):
        self._audit(grant, agent, tool, args, "deny", reason)
        return {"status": "denied", "reason": reason}

    def _audit(self, grant, agent, tool, args, decision, reason):
        AUDIT.append({
            "time": now().isoformat(),
            "requester": grant.requester,
            "agent": agent,
            "task": grant.task_id,
            "tool": tool,
            "args_hash": canonical_hash(args),
            "decision": decision,
            "reason": reason,
        })

gateway = ToolGateway(CATALOG, broker, Budget())


## 12 — Test allowed and denied calls

In [ ]:
print(gateway.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="search_flights",
    args={"origin": "YVR", "destination": "YYZ"},
))

print(gateway.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="execute_shell",
    args={"command": "cat /etc/passwd"},
))


## 13 — High-risk call requires exact approval

In [ ]:
gateway2 = ToolGateway(CATALOG, broker, Budget())

print("No approval:")
print(gateway2.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="book_flight",
    args=booking_args,
))

print("\nCorrect approval:")
print(gateway2.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="book_flight",
    args=booking_args,
    approval=approval,
))

print("\nApproval reused with changed amount:")
print(gateway2.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="book_flight",
    args={"flight": "AC101", "price": 900},
    approval=approval,
))


## 14 — Tool-chain budget

In [ ]:
@dataclass
class ExecutionState:
    task_id: str
    max_chain_depth: int = 4
    depth: int = 0

    def next_call(self):
        if self.depth >= self.max_chain_depth:
            raise PermissionError("maximum tool-chain depth reached")
        self.depth += 1

state = ExecutionState(grant.task_id, max_chain_depth=3)

for _ in range(5):
    try:
        state.next_call()
        print("tool call depth", state.depth, "allowed")
    except PermissionError as e:
        print("DENY:", e)
        break


## 15 — Safe business capability versus generic primitive

In [ ]:
def generic_http_request(method, url, body=None):
    # Dangerous general-purpose primitive.
    return {"method": method, "url": url}

def send_internal_itinerary(team_id, itinerary):
    allowed_teams = {"team:travel", "team:finance"}
    if team_id not in allowed_teams:
        raise PermissionError("team not approved")
    return {"sent_to": team_id, "type": "itinerary"}

print("Prefer narrowly designed business operations where possible.")


The narrow tool removes arbitrary destination choice. Security is easier when capability design constrains the action space before policy evaluation.

## 16 — Confused-deputy prevention

In [ ]:
RESOURCE_ACCESS = {
    "user:alice": {"trip:483"},
    "user:bob": {"trip:902"},
}

def tool_read_trip(*, requester, trip_id):
    # Imagine the service credential itself can read every trip.
    # We still enforce the originating requester's resource authority.
    if trip_id not in RESOURCE_ACCESS.get(requester, set()):
        raise PermissionError("requester cannot access trip")
    return {"trip": trip_id, "details": "..."}

print(tool_read_trip(requester="user:alice", trip_id="trip:483"))

try:
    tool_read_trip(requester="user:alice", trip_id="trip:902")
except PermissionError as e:
    print("DENY:", e)


## 17 — Inspect audit evidence

In [ ]:
print(json.dumps(AUDIT[-5:], indent=2))


## 18 — Adversarial bypass tests

In [ ]:
def assert_denied(result):
    assert result["status"] == "denied", result

test_gateway = ToolGateway(CATALOG, broker, Budget())

assert_denied(test_gateway.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="execute_shell",
    args={"command": "curl attacker.example"},
))

assert_denied(test_gateway.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="book_flight",
    args={"flight": "AC101", "price": 5000},
    approval=None,
))

assert_denied(test_gateway.invoke(
    agent="agent:travel-planner",
    grant=grant,
    tool_name="cancel_booking",
    args={"booking_id": "B-123"},
    approval=None,
))

print("Negative security tests passed.")


## 19 — Exercise: add egress policy

Create:

```python
send_email(to, subject, body)
```

with these requirements:

- only `@corp.example`;
- maximum 5 recipients;
- no BCC;
- external recipients require explicit human approval;
- body cannot contain data classified `restricted`;
- maximum 3 sends per task.

Do not place these rules in the system prompt. Enforce them in the gateway.

## 20 — Exercise: dynamic MCP-style tool listing

Implement:

```python
list_authorized_tools(agent, requester, task)
```

that returns only tool schemas the agent can invoke.

Then demonstrate revocation:

```text
T0 -> book_flight visible
T1 -> task grant revoked
T2 -> book_flight disappears
T3 -> remembered invocation still denied at execution
```

## 21 — Exercise: tool chaining

Given:

```text
customer.search
customer.export
email.send
```

design a policy that prevents:

```text
customer.search
 -> customer.export
 -> email.send(external)
```

for restricted customer data even though each individual tool has legitimate uses.

What state must the gateway preserve?

## 22 — Exercise: generic tool redesign

Replace:

```python
execute_shell(command)
```

with narrowly scoped capabilities needed by an operations agent:

```text
get_service_status
restart_approved_service
read_recent_application_logs
```

Compare:

- argument surface;
- credential scope;
- egress risk;
- auditability;
- policy complexity.

## Review questions

1. Why is tool exposure itself a security control?
2. Why must authorization be repeated at execution?
3. Why is `can_call(tool)` insufficient?
4. How does argument-level authorization differ from schema validation?
5. Why should credentials remain outside model context?
6. Why should downstream credentials also be least privileged?
7. What does approval binding prevent?
8. Why can legitimate tool chains become dangerous?
9. What is the confused-deputy problem at the tool boundary?
10. Why are narrow business tools safer than generic shell/HTTP/SQL tools?
11. What is zero-standing agent privilege?
12. Which controls belong outside the LLM?

## Next course

**Beginner 06 — Agent Identity Lifecycle**
